# DAS Baseline with PCA-Prefix Sites (Base-10 One-Hot Addition)

This notebook evaluates a DAS-style alignment sweep using fixed PCA-rotation interventions.

Task and representation:
- Two-digit base-10 addition
- Inputs are 4 one-hot digits (`A1,B1,A2,B2`) concatenated to width 40
- Output classes are `0..198`

Intervention search space:
- Every hidden layer
- Every prefix size `i in 1..k`
- Subspace is always the PCA-prefix coordinates `[0, 1, ..., i-1]`


## Contents

1. Setup and imports
2. Base-10 SCM definition
3. Load pretrained variable-width MLP checkpoint
4. Build counterfactual dataset (single abstract-variable swap per example)
5. Build PCA-prefix site catalog `(layer, i)`
6. Run DAS-style sweep and report best sites per variable


## Setup


In [ ]:
import os
import random
import itertools
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from pyvene import CausalModel
from pyvene import IntervenableModel
from pyvene import RepresentationConfig, IntervenableConfig
from pyvene import PCARotatedSpaceIntervention


In [ ]:
# Reproducibility and device setup.
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


## Base-10 Addition SCM

Low-digit-first semantics:
- `S1, C1 = add(A1, B1)`
- `S2, C2 = add(A2, B2, C1)`
- `O = 100*C2 + 10*S2 + S1`


In [ ]:
one_hot_digits = [np.eye(10, dtype=np.float32)[d] for d in range(10)]


def as_digit(x):
    arr = np.array(x).reshape(-1)
    if arr.size == 1:
        return int(arr[0])
    return int(arr.argmax())


variables = ["A1", "B1", "A2", "B2", "S1", "C1", "T2", "S2", "C2", "O"]

values = {
    "A1": one_hot_digits,
    "B1": one_hot_digits,
    "A2": one_hot_digits,
    "B2": one_hot_digits,
    "S1": list(range(10)),
    "C1": [0, 1],
    "T2": list(range(19)),
    "S2": list(range(10)),
    "C2": [0, 1],
    "O": list(range(199)),
}

parents = {
    "A1": [],
    "B1": [],
    "A2": [],
    "B2": [],
    "S1": ["A1", "B1"],
    "C1": ["A1", "B1"],
    "T2": ["A2", "B2"],
    "S2": ["T2", "C1"],
    "C2": ["T2", "C1"],
    "O": ["C2", "S2", "S1"],
}


def FILLER():
    return one_hot_digits[0]


functions = {
    "A1": FILLER,
    "B1": FILLER,
    "A2": FILLER,
    "B2": FILLER,
    "S1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) % 10,
    "C1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) // 10,
    "T2": lambda a2, b2: as_digit(a2) + as_digit(b2),
    "S2": lambda t2, c1: (int(t2) + int(c1)) % 10,
    "C2": lambda t2, c1: (int(t2) + int(c1)) // 10,
    "O": lambda c2, s2, s1: int(100 * int(c2) + 10 * int(s2) + int(s1)),
}

addition_model = CausalModel(variables, values, parents, functions)


Quick SCM sanity check


In [ ]:
addition_model.print_structure()

example = {
    "A1": one_hot_digits[9],
    "B1": one_hot_digits[8],
    "A2": one_hot_digits[9],
    "B2": one_hot_digits[9],
}
setting = addition_model.run_forward(example)
print({k: int(setting[k]) for k in ["S1", "C1", "S2", "C2", "O"]})
assert int(setting["O"]) == 197


## Load Pretrained Variable-Width MLP


In [ ]:
MODEL_CHECKPOINT_PATH = "artifacts/addition_variable_width_mlp.pt"


class VariableWidthMLPConfig:
    def __init__(
        self,
        input_dim=40,
        hidden_dims=None,
        num_classes=199,
        dropout=0.0,
        activation="relu",
        include_bias=True,
        squeeze_output=True,
    ):
        if hidden_dims is None:
            hidden_dims = [64, 48, 32]
        self.input_dim = int(input_dim)
        self.hidden_dims = [int(d) for d in hidden_dims]
        self.n_layer = len(self.hidden_dims)
        self.h_dim = self.hidden_dims[-1]
        self.num_classes = int(num_classes)
        self.pdrop = float(dropout)
        self.activation_function = str(activation)
        self.include_bias = bool(include_bias)
        self.squeeze_output = bool(squeeze_output)
        self.problem_type = "single_label_classification"


class VariableWidthMLPBlock(nn.Module):
    def __init__(self, in_dim, out_dim, activation, dropout, include_bias):
        super().__init__()
        self.ff = nn.Linear(in_dim, out_dim, bias=include_bias)
        self.dropout = nn.Dropout(dropout)
        if activation == "relu":
            self.act = nn.ReLU()
        elif activation == "gelu":
            self.act = nn.GELU()
        elif activation == "tanh":
            self.act = nn.Tanh()
        else:
            raise ValueError(f"Unsupported activation: {activation}")

    def forward(self, hidden_states):
        return self.dropout(self.act(self.ff(hidden_states)))


class VariableWidthMLPForClassification(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.h = nn.ModuleList()

        in_dim = config.input_dim
        for out_dim in config.hidden_dims:
            self.h.append(
                VariableWidthMLPBlock(
                    in_dim=in_dim,
                    out_dim=out_dim,
                    activation=config.activation_function,
                    dropout=config.pdrop,
                    include_bias=config.include_bias,
                )
            )
            in_dim = out_dim

        self.score = nn.Linear(in_dim, config.num_classes, bias=config.include_bias)

    def forward(self, input_ids=None, inputs_embeds=None, labels=None, **kwargs):
        hidden_states = inputs_embeds if inputs_embeds is not None else input_ids
        if hidden_states.ndim == 2:
            hidden_states = hidden_states.unsqueeze(1)

        for block in self.h:
            hidden_states = block(hidden_states)

        logits = self.score(hidden_states).squeeze(1)

        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, self.config.num_classes), labels.view(-1))
            return (loss, logits)

        return (logits,)


def logits_from_output(output):
    if isinstance(output, tuple):
        return output[0]
    if hasattr(output, "logits"):
        return output.logits
    return output


if not os.path.exists(MODEL_CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found at {MODEL_CHECKPOINT_PATH}. Run addition_train.ipynb first."
    )

checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=device)
cfg = VariableWidthMLPConfig(**checkpoint["model_config"])
trained = VariableWidthMLPForClassification(cfg).to(device)
trained.load_state_dict(checkpoint["model_state_dict"])
trained.eval()

print("Loaded checkpoint:", MODEL_CHECKPOINT_PATH)
print("Hidden dims:", cfg.hidden_dims)
print("n_layer:", cfg.n_layer)


## Counterfactual Dataset (Single Variable Swap per Example)


In [ ]:
TARGET_VARS = ["S1", "C1", "S2", "C2"]
VAR_VALUE_SPACE = {
    "S1": list(range(10)),
    "C1": [0, 1],
    "S2": list(range(10)),
    "C2": [0, 1],
}
VAR_TO_ID = {v: i for i, v in enumerate(TARGET_VARS)}
ID_TO_VAR = {i: v for v, i in VAR_TO_ID.items()}


def digit_assignment(a1, b1, a2, b2):
    return {
        "A1": one_hot_digits[int(a1)],
        "B1": one_hot_digits[int(b1)],
        "A2": one_hot_digits[int(a2)],
        "B2": one_hot_digits[int(b2)],
    }


# Enumerate full input space once, then bucket by high-level variable values.
all_assignments = [
    digit_assignment(a1, b1, a2, b2)
    for a1, b1, a2, b2 in itertools.product(range(10), repeat=4)
]

bucket_by_var_value = {v: {val: [] for val in VAR_VALUE_SPACE[v]} for v in TARGET_VARS}
for assignment in all_assignments:
    setting = addition_model.run_forward(assignment)
    for var in TARGET_VARS:
        val = int(setting[var])
        bucket_by_var_value[var][val].append(assignment)


# Encodes which abstract variable is being swapped in each example.
def intervention_id(intervention):
    if len(intervention) != 1:
        raise ValueError(f"Expected exactly one swapped variable, got {intervention}")
    var = next(iter(intervention.keys()))
    return VAR_TO_ID[var]


def counterfactual_input_sampler(*args, **kwargs):
    output_var = kwargs.get("output_var", None)
    output_var_value = kwargs.get("output_var_value", None)

    if output_var is None:
        return random.choice(all_assignments)

    return random.choice(bucket_by_var_value[output_var][int(output_var_value)])


def cf_intervention_sampler(*args, **kwargs):
    var = random.choice(TARGET_VARS)
    value = random.choice(VAR_VALUE_SPACE[var])
    return {var: value}


CF_BATCH_SIZE = 128
CF_TRAIN_SIZE = 16000
CF_TEST_SIZE = 6000

train_cf_dataset = addition_model.generate_counterfactual_dataset(
    CF_TRAIN_SIZE,
    intervention_id,
    CF_BATCH_SIZE,
    device=device,
    sampler=counterfactual_input_sampler,
    intervention_sampler=cf_intervention_sampler,
)

print("Counterfactual train dataset size:", len(train_cf_dataset))
print("Example keys:", train_cf_dataset[0].keys())


In [ ]:
# Peek into one counterfactual item.
print("input_ids shape:", tuple(train_cf_dataset[0]["input_ids"].shape))
print("source_input_ids shape:", tuple(train_cf_dataset[0]["source_input_ids"].shape))
print("base_labels:", train_cf_dataset[0]["base_labels"])
print("labels (counterfactual target):", train_cf_dataset[0]["labels"])
print("intervention_id:", train_cf_dataset[0]["intervention_id"], "=>", ID_TO_VAR[int(train_cf_dataset[0]["intervention_id"])])


## Alignment Hyperparameters


In [ ]:
# Edit this cell to test different PCA-prefix alignments.

# Layer subset to search. None => all layers.
SEARCH_LAYER_INDICES = None

# Each searched layer contributes sites i=1..PCA_PREFIX_K.
# Effective max per layer is min(PCA_PREFIX_K, layer_hidden_width).
PCA_PREFIX_K = 8

# Intervention gather/scatter location for sequence models.
# Here the MLP sequence length is 1, so position must stay 0.
ALIGN_COMPONENT = "block_output"
ALIGN_UNIT = "pos"
ALIGN_MAX_UNITS = 1
ALIGN_INTERVENTION_POS = 0

# Number of factual examples used to fit PCA basis per layer.
N_PCA_FIT_EXAMPLES = 20000

print("Search layers:", SEARCH_LAYER_INDICES if SEARCH_LAYER_INDICES is not None else "all")
print("PCA_PREFIX_K:", PCA_PREFIX_K)


In [ ]:
@dataclass
class SiteSpec:
    layer: int
    prefix_dim: int
    hidden_dim: int
    component: str
    unit: str
    position: int
    max_units: int
    pca: object
    pca_mean: np.ndarray
    pca_std: np.ndarray


def factual_input_sampler():
    return random.choice(all_assignments)


def collect_layer_activations(model, inputs_flat, layer_idx, batch_size=256):
    acts = []
    n = inputs_flat.shape[0]
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            hidden = inputs_flat[start:end].to(device).unsqueeze(1)
            for i, block in enumerate(model.h):
                hidden = block(hidden)
                if i == layer_idx:
                    acts.append(hidden[:, 0, :].detach().cpu())
                    break
    return torch.cat(acts, dim=0).numpy()


def fit_layer_pca(acts):
    mean = acts.mean(axis=0)
    std = acts.std(axis=0) + 1e-6
    pca = PCA(n_components=acts.shape[1], svd_solver="full")
    pca.fit((acts - mean) / std)
    return pca, mean, std


def build_site_specs(model, pca_inputs_flat, prefix_k, search_layers=None):
    hidden_dims = list(model.config.hidden_dims)
    layer_ids = list(range(len(hidden_dims))) if search_layers is None else [int(x) for x in search_layers]

    specs = []
    for layer_i in layer_ids:
        width = int(hidden_dims[layer_i])
        acts = collect_layer_activations(model, pca_inputs_flat, layer_i)
        pca, mean, std = fit_layer_pca(acts)

        max_prefix = min(int(prefix_k), width)
        for i in range(1, max_prefix + 1):
            specs.append(
                SiteSpec(
                    layer=layer_i,
                    prefix_dim=i,
                    hidden_dim=width,
                    component=f"h[{layer_i}].output",
                    unit=ALIGN_UNIT,
                    position=ALIGN_INTERVENTION_POS,
                    max_units=ALIGN_MAX_UNITS,
                    pca=pca,
                    pca_mean=mean,
                    pca_std=std,
                )
            )
    return specs


# Build PCA-fit input matrix from factual samples.
pca_examples = addition_model.generate_factual_dataset(N_PCA_FIT_EXAMPLES, factual_input_sampler)
pca_inputs = torch.stack([ex["input_ids"] for ex in pca_examples]).to(torch.float32)

site_specs = build_site_specs(
    trained,
    pca_inputs,
    prefix_k=PCA_PREFIX_K,
    search_layers=SEARCH_LAYER_INDICES,
)

print("Number of candidate sites:", len(site_specs))
print("First few sites:", site_specs[: min(8, len(site_specs))])


## DAS-Style Sweep with PCA-Prefix Interventions


In [ ]:
def build_intervenable_for_site(site):
    pca_intervention = PCARotatedSpaceIntervention(
        embed_dim=site.hidden_dim,
        pca=site.pca,
        pca_mean=site.pca_mean,
        pca_std=site.pca_std,
    )

    cfg_int = IntervenableConfig(
        model_type=type(trained),
        representations=[
            RepresentationConfig(
                layer=site.layer,
                component=site.component,
                unit=site.unit,
                max_number_of_units=site.max_units,
                intervention=pca_intervention,
            )
        ],
    )

    intervenable = IntervenableModel(cfg_int, trained, use_fast=True)
    intervenable.set_device(device)
    intervenable.disable_model_gradients()
    intervenable.disable_intervention_gradients()
    return intervenable


def run_cf_batch_for_site(intervenable, batch, site):
    bsz = batch["input_ids"].shape[0]

    base_batch = batch["input_ids"]
    if base_batch.ndim == 2:
        base_batch = base_batch.unsqueeze(1)

    # source_input_ids can be either [bsz, input_dim] or [bsz, n_sources, input_dim].
    source_raw = batch["source_input_ids"]
    if source_raw.ndim == 2:
        source_batch = source_raw.unsqueeze(1)
    elif source_raw.ndim == 3:
        source_batch = source_raw[:, 0]
        if source_batch.ndim == 2:
            source_batch = source_batch.unsqueeze(1)
    else:
        raise ValueError(f"Unexpected source_input_ids shape: {tuple(source_raw.shape)}")

    pos_list = [[site.position]] * bsz
    prefix_subspace = [list(range(site.prefix_dim))] * bsz

    _, cf_outputs = intervenable(
        {"inputs_embeds": base_batch},
        [{"inputs_embeds": source_batch}],
        {"sources->base": ([pos_list], [pos_list])},
        subspaces=[prefix_subspace],
    )
    return cf_outputs


def evaluate_sites_on_dataset(dataset, site_specs, batch_size):
    # Matrix shape [num_variables, num_sites].
    correct = np.zeros((len(TARGET_VARS), len(site_specs)), dtype=np.float64)
    total = np.zeros((len(TARGET_VARS), len(site_specs)), dtype=np.float64)

    for site_idx, site in enumerate(tqdm(site_specs, desc="Site sweep")):
        intervenable = build_intervenable_for_site(site)

        loader = DataLoader(dataset, batch_size=batch_size)
        with torch.no_grad():
            for batch in loader:
                for k, v in batch.items():
                    if isinstance(v, torch.Tensor):
                        batch[k] = v.to(device)

                cf_outputs = run_cf_batch_for_site(intervenable, batch, site)
                logits = logits_from_output(cf_outputs)

                preds = torch.argmax(logits, dim=1)
                gold = batch["labels"].to(torch.long).view(-1)
                var_ids = batch["intervention_id"].to(torch.long).view(-1)

                for var_idx in range(len(TARGET_VARS)):
                    mask = (var_ids == var_idx)
                    if mask.any():
                        correct[var_idx, site_idx] += float((preds[mask] == gold[mask]).sum().item())
                        total[var_idx, site_idx] += float(mask.sum().item())

    acc = np.divide(correct, np.maximum(total, 1.0))
    return acc


In [ ]:
# Held-out counterfactual dataset for reporting search results.
test_cf_dataset = addition_model.generate_counterfactual_dataset(
    CF_TEST_SIZE,
    intervention_id,
    CF_BATCH_SIZE,
    device=device,
    sampler=counterfactual_input_sampler,
    intervention_sampler=cf_intervention_sampler,
)

acc_matrix = evaluate_sites_on_dataset(test_cf_dataset, site_specs, batch_size=CF_BATCH_SIZE)
print("Accuracy matrix shape [variables, sites]:", acc_matrix.shape)


In [ ]:
site_labels = [f"L{site.layer}-i{site.prefix_dim}" for site in site_specs]

plt.figure(figsize=(max(10, 0.45 * len(site_specs)), 3.8))
plt.imshow(acc_matrix, cmap="viridis", aspect="auto", vmin=0.0, vmax=1.0)
plt.colorbar(label="counterfactual accuracy")
plt.yticks(range(len(TARGET_VARS)), TARGET_VARS)
plt.xticks(range(len(site_labels)), site_labels, rotation=90)
plt.xlabel("Neural site (layer, PCA-prefix size)")
plt.ylabel("Abstract variable")
plt.title("DAS-style PCA-prefix sweep")
plt.tight_layout()
plt.show()

for var_idx, var in enumerate(TARGET_VARS):
    best_site_idx = int(np.argmax(acc_matrix[var_idx]))
    best_site = site_specs[best_site_idx]
    print(
        f"{var}: best site = L{best_site.layer}-i{best_site.prefix_dim} "
        f"(hidden={best_site.hidden_dim}), accuracy={acc_matrix[var_idx, best_site_idx]:.4f}"
    )


## Notes

- This notebook uses fixed PCA rotations and sweeps `(layer, prefix-size)` combinations, so there is no learned rotation step.
- If you want a narrower search, edit `SEARCH_LAYER_INDICES` and `PCA_PREFIX_K` in the hyperparameter cell and re-run from there.
